# Gnani Prisma (ASR) & Timbre (TTS) Evaluation

**Engineer:** Ashish Vipin  
**Role:** AI Solutions Engineer Evaluation  
**Target Systems:** Gnani Prisma v3.0 (ASR) & Timbre v2.5 (TTS) via api.vachana.ai

---

## Overview
This notebook executes an end-to-end benchmarking harness evaluating:
1. **Prisma ASR**: 30 test utterances spanning 11 categories (Clean, Noisy, Telephony, Numbers/ITN, Code-Switching, Accents, Domain-Specific, Length variations, and Regional Indic languages).
2. **Timbre TTS**: 28 test sentences spanning 9 categories evaluated across Latency, RTF, Speaking Rate, Silence Ratio, Clipping, and Round-Trip ASR transcription fidelity.
3. **Metric Implementation**: WER & CER calculated from scratch via Wagner-Fischer dynamic programming algorithm.

In [1]:
import os
import sys
import json
from pathlib import Path

# Add current directory to path
sys.path.insert(0, str(Path.cwd()))

from config import GNANI_API_KEY, ASR_MODEL, TTS_MODEL
from src.metrics import compute_wer, compute_cer, aggregate_wer, aggregate_cer
from src.gnani_client import GnaniClient

print('API Key Configured:', 'Yes' if GNANI_API_KEY else 'No')
print(f'Target Models: ASR={ASR_MODEL}, TTS={TTS_MODEL}')

API Key Configured: Yes
Target Models: ASR=prisma-v3.0, TTS=timbre-v2.5


## Part 1: Metric Verification (Scratch Levenshtein DP)
Verifying our custom Levenshtein implementation with substitution (S), deletion (D), and insertion (I) breakdown.

In [2]:
sample_ref = 'the cat sat on the mat'
sample_hyp = 'the cat sit on a mat'

res_wer = compute_wer(sample_ref, sample_hyp)
res_cer = compute_cer(sample_ref, sample_hyp)

print('Reference :', sample_ref)
print('Hypothesis:', sample_hyp)
print(f"WER: {res_wer['wer']:.2%} (S={res_wer['substitutions']}, D={res_wer['deletions']}, I={res_wer['insertions']})")
print(f"CER: {res_cer['cer']:.2%}")

Reference : the cat sat on the mat
Hypothesis: the cat sit on a mat
WER: 33.33% (S=2, D=0, I=0)
CER: 23.53%


## Part 2: Prisma ASR Empirical Benchmark Results
Loading and reviewing the evaluation results generated against Prisma v3.0.

In [3]:
with open('results/asr_results.json', 'r', encoding='utf-8') as f:
    asr_res = json.load(f)

print('=' * 50)
print(f"PRISMA ASR OVERALL BENCHMARK (n={asr_res['metadata']['total_clips']})")
print(f"Overall Micro-averaged WER: {asr_res['overall']['wer']:.2%}")
print(f"Overall Micro-averaged CER: {asr_res['overall']['cer']:.2%}")
print('=' * 50)

header = f"{'Category':<26} | {'WER':<8} | {'CER':<8} | {'Avg Latency':<12} | {'Clips'}"
print(header)
print('-' * len(header))
for cat, data in asr_res['by_category'].items():
    print(f"{cat:<26} | {data['aggregate_wer']*100:>6.2f}% | {data['aggregate_cer']*100:>6.2f}% | {data['avg_latency_ms']:>8.1f} ms | {data['num_clips']:>5}")

PRISMA ASR OVERALL BENCHMARK (n=30)
Overall Micro-averaged WER: 20.67%
Overall Micro-averaged CER: 18.99%
Category                   | WER      | CER      | Avg Latency  | Clips
-----------------------------------------------------------------------
clean_english              |   9.68% |  12.16% |    592.3 ms |     3
clean_hindi                |   4.55% |   8.05% |   1084.6 ms |     3
noisy_audio                |  18.75% |  16.00% |    380.5 ms |     3
telephony                  |   6.25% |   5.00% |    269.2 ms |     2
numbers_and_entities       |  66.67% |  68.10% |    421.5 ms |     3
code_switching             |  38.46% |  33.88% |    354.3 ms |     3
accented_english           |   0.00% |   0.00% |    291.1 ms |     2
domain_specific            |  24.24% |  13.37% |    342.0 ms |     3
long_utterance             |   9.84% |  11.11% |    796.8 ms |     2
short_utterance            |   0.00% |   0.00% |    127.6 ms |     3
regional_languages         |   6.25% |   4.12% |    258.1 ms

### Qualitative Error Analysis: The Inverse Text Normalisation (ITN) Trade-off
Inspect how Prisma normalises numbers, entities, and code-mixed terms into production-ready representations.

In [4]:
for r in asr_res['per_clip_results']:
    if r['category'] in ['numbers_and_entities', 'code_switching', 'telephony']:
        print(f"ID: {r['id']} ({r['category']})")
        print(f"  Ref: {r['reference']}")
        print(f"  Hyp: {r['hypothesis']}")
        print(f"  WER: {r['wer']*100:.1f}% | CER: {r['cer']*100:.1f}%")
        print('-' * 40)

ID: telephony_en_001 (telephony)
  Ref: can you please repeat your account number
  Hyp: Can you please repeat your account number?
  WER: 0.0% | CER: 0.0%
----------------------------------------
ID: telephony_hi_001 (telephony)
  Ref: आपका कॉल हमारे लिए महत्वपूर्ण है कृपया प्रतीक्षा करें
  Hyp: आपका call हमारे लिए महत्वपूर्ण है, कृपया प्रतीक्षा करें।
  WER: 11.1% | CER: 8.9%
----------------------------------------
ID: numbers_en_001 (numbers_and_entities)
  Ref: the total amount is one lakh twenty three thousand four hundred and fifty six rupees
  Hyp: The total amount is 1,23,456 rupees.
  WER: 66.7% | CER: 68.6%
----------------------------------------
ID: numbers_en_002 (numbers_and_entities)
  Ref: my mobile number is nine eight seven six five four three two one zero
  Hyp: My mobile number is 9876543210.
  WER: 71.4% | CER: 71.4%
----------------------------------------
ID: numbers_hi_001 (numbers_and_entities)
  Ref: कुल राशि पाँच लाख बहत्तर हज़ार तीन सौ रुपये है
  Hyp: कुल रा

## Part 3: Timbre TTS Quality & Latency Benchmark Results
Inspecting the synthesis latency, Real-Time Factor (RTF), speaking rate, silence ratio, and automated quality scoring.

In [5]:
with open('results/tts_results.json', 'r', encoding='utf-8') as f:
    tts_res = json.load(f)

print('=' * 50)
print(f"TIMBRE TTS OVERALL BENCHMARK (n={tts_res['metadata']['total_sentences']})")
print(f"Average Quality Score: {tts_res['overall']['avg_quality_score']:.1f}/100")
print(f"Average Real-Time Factor (RTF): {tts_res['overall']['avg_rtf']:.4f}")
print(f"Average Latency: {tts_res['overall']['avg_latency_ms']:.1f} ms")
print(f"Latency Range: {tts_res['overall']['min_latency_ms']:.1f} ms - {tts_res['overall']['max_latency_ms']:.1f} ms")
print('=' * 50)

header = f"{'Category':<26} | {'Score':<8} | {'RTF':<8} | {'Avg Latency':<12} | {'WPM':<6}"
print(header)
print('-' * len(header))
for cat, data in tts_res['by_category'].items():
    print(f"{cat:<26} | {data['avg_quality_score']:>6.1f} | {data['avg_rtf']:>8.4f} | {data['avg_latency_ms']:>8.1f} ms | {data['avg_speaking_rate_wpm']:>6.1f}")

TIMBRE TTS OVERALL BENCHMARK (n=28)
Average Quality Score: 88.0/100
Average Real-Time Factor (RTF): 0.3257
Average Latency: 1501.8 ms
Latency Range: 525.2 ms - 7541.0 ms
Category                   | Score    | RTF      | Avg Latency  | WPM   
------------------------------------------------------------------------
numbers                    |   78.8 |   0.6547 |   2556.5 ms |   90.3
abbreviations              |   91.8 |   0.2399 |   1243.6 ms |  133.4
code_switching             |   82.7 |   0.2595 |   1010.0 ms |  179.7
long_form                  |   96.5 |   0.1000 |   2005.5 ms |  149.1
tricky_pronunciation       |   92.5 |   0.5231 |   1852.3 ms |  170.4
punctuation_and_prosody    |   96.0 |   0.1611 |    876.6 ms |  150.6
special_formats            |   88.7 |   0.2653 |   1675.5 ms |  110.0
regional_languages         |   93.0 |   0.1926 |    737.5 ms |  125.2
edge_cases                 |   79.3 |   0.2134 |    881.4 ms |  206.8


## Part 4: Interactive Test Harness
Call Prisma ASR or Timbre TTS on any custom utterance or sentence on-the-fly.

In [6]:
client = GnaniClient()

# Test TTS generation
test_text = 'नमस्ते, सनराइज़ बैंक में आपका स्वागत है। आपका अकाउंट बैलेंस 50,000 रुपये है।'
output_audio = 'data/tts/output/demo_sample.wav'

tts_out = client.synthesize(text=test_text, language='hi-IN', output_path=output_audio)
print(f"Synthesized in {tts_out['latency_ms']} ms -> {output_audio}")

# Transcribe it right back with Prisma ASR
asr_out = client.transcribe(output_audio, language='hi-IN')
print(f"Transcribed in {asr_out['latency_ms']} ms:")
print(f"Output: {asr_out['transcript']}")

Synthesized in 3493.83 ms -> data/tts/output/demo_sample.wav
Transcribed in 369.98 ms:
Output: नमस्ते, Sunrise Bank में आपका स्वागत है। आपका account balance 50,000 रुपये है।
